<div style="border-left: 5px solid #b7791f; background-color: #fff8e1; padding: 0.8em 1em; margin: 1em 0; border-radius: 4px;">
  <strong>Warning: AI-assisted materials</strong><br><br>
  These materials were developed with assistance from AI tools. All content has been reviewed and edited by the instructor, who takes final responsibility for its accuracy, clarity, and appropriateness for the course. Students should treat these materials as instructor-reviewed course content while applying the same critical judgment they would use with any technical material. Please report any suspected errors or unclear explanations to ghunt@wm.edu.
</div>

# Multiclass Logistic Regression

Multiclass logistic regression extends the binary model to $K>2$ classes. The ERM structure is unchanged: construct linear scores, convert them to probabilities, and minimize cross-entropy loss. Let
$$
y\in\{1,\ldots,K\},\qquad x\in\mathbb R^D.
$$
We use one weight vector and score for each class:
$$
s_k(x)=w_k^\top x,\qquad k=1,\ldots,K.
$$
A larger value of $s_k(x)$ means that the model gives more support to class $k$. These scores can be any real numbers, so they are not themselves probabilities.

Collect the weights as columns of $W=(w_1,\ldots,w_K)\in\mathbb R^{D\times K}$ so that
$$
W =
\begin{pmatrix}
\vert &        & \vert \\
w_1   & \cdots & w_K   \\
\vert &        & \vert
\end{pmatrix}
\in \mathbb{R}^{D \times K}
$$
is the parameter matrix whose $k$-th column is $w_k \in \mathbb{R}^D$.


Then all $K$ scores can be computed at once:
$$
s_W(x)=W^\top x\in\mathbb R^K,
$$
whose $k$th entry is $w_k^\top x$. The prediction action selects the class with the largest score:
$$
\hat f(x)=\arg\max_{1\le k\le K}\hat w_k^\top x.
$$

## Softmax and Cross-Entropy

The **softmax** map converts the score vector directly into probabilities:
$$
p_{W,k}(x)=\operatorname{softmax}_k(W^\top x)=\frac{e^{s_k(x)}}{\sum_{j=1}^K e^{s_j(x)}}
=\frac{e^{w_k^\top x}}{\sum_{j=1}^K e^{w_j^\top x}}.
$$
If we collect together all $k$ components we can write:
$$
p_W(x)=\operatorname{softmax}(W^\top x)=\left(\frac{e^{s_1(x)}}{\sum_{j=1}^K e^{s_j(x)}},\ldots,\frac{e^{s_K(x)}}{\sum_{j=1}^K e^{s_j(x)}}\right)^\top
$$
Every $p_{W,k}(x)$ is positive, and the common denominator ensures that
$$
\sum_{k=1}^Kp_{W,k}(x)
=\frac{\sum_{k=1}^K e^{s_k(x)}}{\sum_{j=1}^K e^{s_j(x)}}=1.
$$
The exponential function is increasing, so softmax also preserves the ordering of the scores:
$$
s_a(x)>s_b(x)\quad\Longleftrightarrow\quad p_{W,a}(x)>p_{W,b}(x).
$$
Consequently, the largest score and largest predicted probability identify the same class.

Represent class $y$ by a one-hot vector $t\in\{0,1\}^K$, where $t_y=1$ and all other entries are zero. For example, if $K=3$ and $y=2$, then $t=(0,1,0)^\top$.

The multiclass cross-entropy loss is
$$
L_{\mathrm{CE}}(t,p)=-\sum_{k=1}^K t_k\log p_k=-\log p_y.
$$
where the true class is $y$. The one-hot vector selects the term associated with the observed class. Thus the loss is small when the model assigns high probability to that class and becomes large as $p_y$ approaches zero.

Substituting the softmax formula expresses the same loss directly in terms of the scores using one-hot notation: 
$$
\ell(t, s_W(x)) =
\log \left( \sum_{j=1}^K e^{w_j^\top x} \right) -
\sum_{k=1}^K t_k w_k^\top x.
$$
or not using one-hot:
$$
\begin{aligned}
\ell(y,s_W(x))
&=-\log\left(\frac{e^{w_y^\top x}}{\sum_{j=1}^K e^{w_j^\top x}}\right)\\
&=\log\left(\sum_{j=1}^K e^{w_j^\top x}\right)-w_y^\top x.
\end{aligned}
$$
where the true class is $y$. Here $\ell(t,s_W(x))$ and $\ell(y,s_W(x))$ denote the same loss, with the class represented in one-hot and label form, respectively.

## ERM in Matrix Form

Suppose we observe $(x_1,y_1),\ldots,(x_N,y_N)$. For observation $n$, let
$$
p_n=p_W(x_n)=\operatorname{softmax}(W^\top x_n),
\qquad
\ell_n=-\log p_{n,y_n}.
$$
The empirical risk averages these observation-level losses:
$$
\widehat R(W)=\frac1N\sum_{n=1}^N\ell_n.
$$

We can write the entire calculation compactly by defining

- $X\in\mathbb R^{N\times D}$, with row $n$ equal to $x_n^\top$
- $T\in\mathbb R^{N\times K}$, with row $n$ equal to the one-hot vector $t_n^\top$
- $S=XW\in\mathbb R^{N\times K}$, the score matrix, with $S_{nk}=s_k(x_n)$
- $P=\operatorname{softmax}(S)\in\mathbb R^{N\times K}$, applying softmax rowwise.

Row $n$ of $S$ contains the $K$ scores for $x_n$, and row $n$ of $P$ contains the corresponding probabilities. Therefore
$$
\widehat R(W)=-\frac1N\sum_{n=1}^N\sum_{k=1}^K T_{nk}\log P_{nk}.
$$
The ERM estimate is
$$
\widehat W=\arg\min_{W\in\mathbb R^{D\times K}}\widehat R(W).
$$
This is the binary logistic-regression objective with scalar probabilities replaced by $K$-class probability vectors.

## Identifiability

Adding the same vector $a\in\mathbb R^D$ to every class weight does not change the model:
$$
w_k' = w_k+a\quad\text{for every }k.
$$
Indeed, every score gains the same value $a^\top x$, and that common factor cancels:
$$
\begin{aligned}
p_{W',k}(x)
&=\frac{e^{w_k^\top x+a^\top x}}
{\sum_j e^{w_j^\top x+a^\top x}}\\
&=\frac{e^{a^\top x}e^{w_k^\top x}}
{e^{a^\top x}\sum_j e^{w_j^\top x}}
=p_{W,k}(x).
\end{aligned}
$$
Thus many different parameter matrices produce exactly the same probabilities and predictions. The identifiable quantities are contrasts such as $w_k-w_j$. One can remove the redundancy by choosing a reference class, for example $w_K=0$; an unconstrained numerical optimizer instead selects one of the equivalent parameter matrices.

## Gradient

For observation $n$, let
$$
\ell_n=-\sum_{k=1}^K t_{nk}\log p_{nk}.
$$
Then
$$
\widehat R(W)=\frac1N\sum_{n=1}^N\ell_n.
$$

We derive the gradient one class at a time. Write the score for class $k$ and observation $n$ as
$$
s_{nk}=s_k(x_n)=w_k^\top x_n.
$$
The softmax cross-entropy calculation gives the useful identity
$$
\frac{\partial\ell_n}{\partial s_{nk}}=p_{nk}-t_{nk}.
$$
Since $\nabla_{w_k}s_{nk}=x_n$, the scalar chain rule gives
$$
\nabla_{w_k}\ell_n=(p_{nk}-t_{nk})x_n.
$$
Averaging over observations produces one gradient vector for each class:
$$
\nabla_{w_k}\widehat R(W)
=\frac1N\sum_{n=1}^N(p_{nk}-t_{nk})x_n
=\frac1N X^\top(P_{:k}-T_{:k}).
$$
Here $P_{:k}$ and $T_{:k}$ denote column $k$ of $P$ and $T$. Stacking these $K$ class gradients as the columns of $\nabla_W\widehat R(W)$ gives
$$
\boxed{\nabla_W\widehat R(W)=\frac1N X^\top(P-T)}.
$$
This has the same residual-times-features structure as the binary result, except that $P-T$ is now an $N\times K$ matrix. The shape check is
$$
(D\times N)(N\times K)=D\times K,
$$
so the gradient has the same shape as $W$.

## Gradient Descent

Setting the gradient equal to zero produces nonlinear equations, so there is no normal-equation-style formula for $\widehat W$. We instead update all class weights together:
$$
W^{(t+1)}=W^{(t)}-\eta\frac1N X^\top(P^{(t)}-T),
\qquad
P^{(t)}=\operatorname{softmax}(XW^{(t)}).
$$
At iteration $t$:

1. compute the score matrix $S^{(t)}=XW^{(t)}$
2. apply softmax rowwise to obtain $P^{(t)}$
3. compute $\nabla_W\widehat R(W^{(t)})=X^\top(P^{(t)}-T)/N$
4. step in the negative-gradient direction to obtain $W^{(t+1)}$.

The learning rate $\eta>0$ controls the step size: a very small value gives slow progress, while a value that is too large can overshoot and prevent convergence. Iterations continue until the loss, gradient, or parameters change very little. This is the same procedure used for binary logistic regression, now applied to a matrix of parameters.

## Decision Boundaries

Any boundary shared by regions predicted as classes $a$ and $b$ lies where their scores tie:
$$
w_a^\top x=w_b^\top x\quad\Longleftrightarrow\quad(w_a-w_b)^\top x=0.
$$
Thus the decision boundaries lie in pairwise hyperplanes in the chosen feature space, which is why this is called a **linear classifier**. Applying the same strictly increasing transformation to every class score would not change their ordering or the predicted class.

## A Small Simulation

We first implement the formulas directly. In the code, classes are indexed by $0,\ldots,K-1$ to follow Python's convention. The simulation includes an intercept as a leading column of ones in $X$; later, scikit-learn fits the intercept separately by default.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def softmax(scores):
    scores = np.asarray(scores)
    shifted = scores - np.max(scores, axis=-1, keepdims=True)
    exps = np.exp(shifted)
    return exps / np.sum(exps, axis=-1, keepdims=True)

def one_hot(y, K):
    T = np.zeros((len(y), K))
    T[np.arange(len(y)), y] = 1
    return T

def softmax_loss(X, T, W):
    P = np.clip(softmax(X @ W), 1e-12, 1.0)
    return -np.sum(T * np.log(P)) / X.shape[0]

def softmax_gradient(X, T, W):
    P = softmax(X @ W)
    return X.T @ (P - T) / X.shape[0]

def softmax_regression_gd(X, T, eta=0.1, max_iter=5000, tol=1e-8):
    W = np.zeros((X.shape[1], T.shape[1]))
    losses = [softmax_loss(X, T, W)]

    for _ in range(max_iter):
        grad = softmax_gradient(X, T, W)
        if np.linalg.norm(grad) < tol:
            break
        W = W - eta * grad
        losses.append(softmax_loss(X, T, W))

    return W, np.array(losses)

In [ ]:
def simulate_softmax_data(n=600, seed=1234):
    rng = np.random.default_rng(seed)
    X_raw = rng.normal(size=(n, 2))
    X = np.column_stack([np.ones(n), X_raw])

    W_true = np.array([
        [ 0.0,  0.4, -0.4],
        [ 1.8, -1.2, -0.3],
        [-0.4,  1.5, -1.5],
    ])

    P = softmax(X @ W_true)
    y = np.array([rng.choice(3, p=P[n]) for n in range(len(X))])
    T = one_hot(y, 3)
    return X, y, T

X_sim, y_sim, T_sim = simulate_softmax_data()
print("X shape:", X_sim.shape)
print("T shape:", T_sim.shape)
print("Class counts:", T_sim.sum(axis=0).astype(int))

In [ ]:
plt.figure(figsize=(5, 5))
for k in range(3):
    plt.scatter(X_sim[y_sim == k, 1], X_sim[y_sim == k, 2], label=f"class {k}")
plt.xlabel(r"$x_1$")
plt.ylabel(r"$x_2$")
plt.title("Simulated three-class data")
plt.legend()
plt.axis("equal")
plt.show()

In [ ]:
W_hat, loss_history = softmax_regression_gd(
    X_sim, T_sim, eta=0.5, max_iter=10000, tol=1e-6
)
P_hat = softmax(X_sim @ W_hat)
y_hat = np.argmax(P_hat, axis=1)

print("Final loss:", loss_history[-1])
print("Training classification error:", np.mean(y_hat != y_sim))

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("Iteration")
plt.ylabel("Cross-entropy loss")
plt.title("Gradient descent")
plt.grid(True)
plt.show()

In [ ]:
#| code-fold: true

def plot_softmax_decision_regions(X, y, W, grid_points=300, pad=0.6):
    x1, x2 = X[:, 1], X[:, 2]
    K = W.shape[1]
    xx1, xx2 = np.meshgrid(
        np.linspace(x1.min() - pad, x1.max() + pad, grid_points),
        np.linspace(x2.min() - pad, x2.max() + pad, grid_points),
    )
    X_grid = np.column_stack([np.ones(xx1.size), xx1.ravel(), xx2.ravel()])
    y_grid = np.argmax(softmax(X_grid @ W), axis=1).reshape(xx1.shape)

    cmap = plt.get_cmap("tab10")
    colors = [cmap(k) for k in range(K)]
    plt.figure(figsize=(6, 5))
    plt.contourf(
        xx1, xx2, y_grid, levels=np.arange(K + 1) - 0.5,
        colors=colors, alpha=0.25,
    )
    for k in range(K):
        plt.scatter(
            x1[y == k], x2[y == k], color=colors[k],
            edgecolor="black", s=35, label=f"class {k}",
        )
    plt.xlabel(r"$x_1$")
    plt.ylabel(r"$x_2$")
    plt.title("Estimated multiclass decision regions")
    plt.legend()
    plt.show()

plot_softmax_decision_regions(X_sim, y_sim, W_hat)

## Penguins

We now fit a standard multiclass logistic-regression implementation to the three-species penguins dataset. Using two measurements lets us see the resulting class regions directly.

In [ ]:
import pandas as pd
import seaborn as sns

from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

sns.set_theme(style="whitegrid")

In [ ]:
url = "https://gist.githubusercontent.com/slopp/ce3b90b9168f2f921784de84fa445651/raw/penguins.csv"
cols = ["flipper_length_mm", "bill_length_mm", "species"]
penguins = (
    pd.read_csv(url)[cols]
    .dropna()
    .sample(frac=1, random_state=1234)
)
X_peng = penguins[["flipper_length_mm", "bill_length_mm"]].to_numpy()
y_peng = penguins["species"].to_numpy()
penguins.head()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(
    data=penguins, x="flipper_length_mm", y="bill_length_mm",
    hue="species", ax=ax,
)
ax.set_title("Penguins: three species")
plt.show()

In [ ]:
linear_mod = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=np.inf, max_iter=10000),
)
linear_mod.fit(X_peng, y_peng)
linear_pred = linear_mod.predict(X_peng)
print("Training accuracy:", np.mean(linear_pred == y_peng))

In [ ]:
#| code-fold: true

def plot_penguin_regions(model, title):
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    DecisionBoundaryDisplay.from_estimator(
        model, X_peng, response_method="predict", alpha=0.25, ax=ax,
    )
    sns.scatterplot(
        data=penguins, x="flipper_length_mm", y="bill_length_mm",
        hue="species", edgecolor="black", linewidth=0.5, ax=ax,
    )
    ax.set_title(title)
    plt.show()

plot_penguin_regions(linear_mod, "Linear multiclass decision regions")

For $K$ classes, the confusion matrix is a $K\times K$ table: rows are true classes, columns are predicted classes, and diagonal entries are correct predictions.

In [ ]:
classes = linear_mod.named_steps["logisticregression"].classes_
cm = confusion_matrix(y_peng, linear_pred, labels=classes)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ConfusionMatrixDisplay(cm, display_labels=classes).plot(ax=ax, colorbar=False)
ax.grid(False)
plt.show()

### Feature Engineering

The classifier is linear in its chosen features. Adding polynomial features therefore permits curved boundaries in the original two-dimensional input space; the fitting procedure itself is unchanged.

In [ ]:
poly_mod = make_pipeline(
    PolynomialFeatures(degree=3, include_bias=False),
    StandardScaler(),
    LogisticRegression(C=np.inf, max_iter=10000),
)
poly_mod.fit(X_peng, y_peng)
plot_penguin_regions(poly_mod, "Decision regions with polynomial features")

## Review Questions

See: @sec-mvlogistic-questions.